Transformer Encoder-Decoder
====


  - encoder self-attention: bidirectional, no causal mask
  - decoder self-attention: causal mask
  - decoder cross-attention: queries from decoder, keys/values from encoder
  - feed-forward network after attention
  - stacked encoder and decoder blocks

https://github.com/huggingface/transformers/blob/main/src/transformers/models/bart/modeling_bart.py

In [1]:
import torch
import torch.nn as nn
import tiktoken

In [2]:
embedding_dim = 768
context_length = 1024
num_heads = 12

tokenizer = tiktoken.get_encoding("gpt2")
n_vocab = tokenizer.n_vocab

print(f"Vocabulary size: {n_vocab}")

Vocabulary size: 50257


### 1. Prepare the input

In [3]:
input_texts = ["Hello, how are you?", "Transformers are amazing!"]
input_ids = [tokenizer.encode(t) for t in input_texts]
print(f"input_ids: {input_ids}")

# apply padding to input_ids to make them the same length
pad_token_id = tokenizer.eot_token  # GPT-2 tokenizer has no dedicated pad token
max_len = max(len(ids) for ids in input_ids)

padded_input_ids = [
    ids + [pad_token_id] * (max_len - len(ids))
    for ids in input_ids
]

input_ids_tensor = torch.tensor(padded_input_ids, dtype=torch.long)

# length-based masking for attention
lengths = torch.tensor([len(ids) for ids in input_ids])
positions = torch.arange(max_len).unsqueeze(0)
attention_mask = positions >= lengths.unsqueeze(1)  # True means padding

print("input_ids_tensor:", input_ids_tensor.shape)  # [batch_size, seq_len]
print("input_ids_tensor:\n", input_ids_tensor)
print("attention_mask:", attention_mask.shape)
print("attention_mask:\n", attention_mask)

input_ids: [[15496, 11, 703, 389, 345, 30], [41762, 364, 389, 4998, 0]]
input_ids_tensor: torch.Size([2, 6])
input_ids_tensor:
 tensor([[15496,    11,   703,   389,   345,    30],
        [41762,   364,   389,  4998,     0, 50256]])
attention_mask: torch.Size([2, 6])
attention_mask:
 tensor([[False, False, False, False, False, False],
        [False, False, False, False, False,  True]])


In [4]:
tok_embeddings = nn.Embedding(
    num_embeddings=n_vocab, embedding_dim=embedding_dim
)

class SinusoidalPositionEmbedding(nn.Module):
    def __init__(self, context_length, embedding_dim):
        super().__init__()
        position = torch.arange(context_length).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, embedding_dim, 2) * (-torch.log(torch.tensor(10000.0)) / embedding_dim)
        )

        pe = torch.zeros(context_length, embedding_dim)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe)

    def forward(self, positions):
        return self.pe[positions]


pos_embeddings = SinusoidalPositionEmbedding(
    context_length=context_length, embedding_dim=embedding_dim
)

tok_emb = tok_embeddings(input_ids_tensor)

batch_size, seq_len = input_ids_tensor.shape

positions = torch.arange(seq_len, dtype=torch.long, device=input_ids_tensor.device)
pos_emb = pos_embeddings(positions)  # [seq_len, embedding_dim]
print(f"pos_emb.shape: {pos_emb.shape}")  # [seq_len, 768]

batch_x = tok_emb + pos_emb

print(f"Input x: {batch_x.shape}")

pos_emb.shape: torch.Size([6, 768])
Input x: torch.Size([2, 6, 768])


### 2. Multi-head Attention (MHA)

In [5]:

class MultiheadAttention(nn.Module):
    def __init__(
            self,
            embedding_dim,
            num_heads,
            context_length,
            is_crossattention=False,
            is_causal=False,
            dropout=0.1
        ):
        super().__init__()
        assert embedding_dim % num_heads == 0, "embedding_dim must be divisible by num_heads"
        self.num_heads = num_heads
        self.embedding_dim = embedding_dim
        self.head_dim = embedding_dim // num_heads
        self.is_crossattention = is_crossattention
        self.is_causal = is_causal

        # if self.is_crossattention:
        #     self.w_attn = nn.Linear(embedding_dim, 2 * embedding_dim)  # for k & v
        #     self.w_q = nn.Linear(embedding_dim, embedding_dim)
        # else:
        #     self.w_attn = nn.Linear(embedding_dim, 3 * embedding_dim)
        self.w_q = nn.Linear(embedding_dim, embedding_dim)
        self.w_k = nn.Linear(embedding_dim, embedding_dim)
        self.w_v = nn.Linear(embedding_dim, embedding_dim)
        self.w_o = nn.Linear(embedding_dim, embedding_dim)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        if self.is_causal:
            mask = torch.triu(
                torch.ones(context_length, context_length, dtype=torch.bool),
                diagonal=1,
            )
            self.register_buffer("causal_mask", mask)

    def forward(self, x, kv_states=None, attention_mask=None):
        batch_size, target_len, _ = x.size()

        if self.is_crossattention:
            assert kv_states is not None, "kv_states must be provided for cross-attention"
        else:
            kv_states = x
        src_len = kv_states.size(1)

        # Linear projections
        # if self.is_crossattention:
        #     q = self.w_q(x)
        #     k, v = self.w_attn(kv_states).split(self.embedding_dim, dim=-1)
        # else:
        #     q, k, v = self.w_attn(x).split(self.embedding_dim, dim=-1)
        q = self.w_q(x)
        k = self.w_k(kv_states)
        v = self.w_v(kv_states)
        # current shapes: [batch_size, seq_len, embedding_dim]

        # split the embedding dimension into multiple heads
        # -> shape: [batch_size, num_heads, seq_len, head_dim]
        q = q.view(batch_size, target_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, src_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, src_len, self.num_heads, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        
        # Causal mask
        if self.is_causal:
            assert target_len <= self.causal_mask.size(0), "sequence length exceeds context_length"
            assert src_len <= self.causal_mask.size(0), "source length exceeds context_length"

            causal_mask = self.causal_mask[:target_len, :src_len]
            attn_scores = attn_scores.masked_fill(
                causal_mask.unsqueeze(0).unsqueeze(0),
                float('-inf')
            )

        # attention mask (for padding tokens)
        if attention_mask is not None:
            assert attention_mask.shape == (batch_size, src_len)
            assert attention_mask.dtype == torch.bool

            # attention_mask shape: [batch_size, src_len]
            attn_scores = attn_scores.masked_fill(
                attention_mask.unsqueeze(1).unsqueeze(2),
                float("-inf"),
            )

        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)
        output = torch.matmul(attn_weights, v)

        # Concatenate heads and project
        output = output.transpose(1, 2).contiguous().view(batch_size, target_len, -1)
        output = self.w_o(output)
        output = self.resid_dropout(output)

        return output
    
# testing:
mha_layer = MultiheadAttention(
    embedding_dim=embedding_dim,
    num_heads=8,
    context_length=context_length,
    dropout=0.1
)
output_mha = mha_layer(batch_x)
print(f"Input shape: {batch_x.shape}")
print(f"MHA Output shape: {output_mha.shape}")

mha_layer_cross = MultiheadAttention(
    embedding_dim=embedding_dim,
    num_heads=8,
    context_length=context_length,
    is_crossattention=True,
    is_causal=False,
    dropout=0.1
)
output_mha_cross = mha_layer_cross(batch_x, kv_states=batch_x)
print(f"MHA (cross-attention) output shape: {output_mha_cross.shape}")

mha_layer_causal = MultiheadAttention(
    embedding_dim=embedding_dim,
    num_heads=8,
    context_length=context_length,
    is_crossattention=False,
    is_causal=True,
    dropout=0.1
)

output_mha_causal = mha_layer_causal(batch_x, attention_mask=attention_mask)
print(f"MHA (causal) output shape: {output_mha_causal.shape}")

Input shape: torch.Size([2, 6, 768])
MHA Output shape: torch.Size([2, 6, 768])
MHA (cross-attention) output shape: torch.Size([2, 6, 768])
MHA (causal) output shape: torch.Size([2, 6, 768])


### 3. Transformer Encoder

In [6]:
# MHA -> dropout -> residual -> layer norm -> FFN -> dropout -> residual -> layer norm

class EncoderBlock(nn.Module):
    def __init__(self, embedding_dim, num_heads, context_length, dropout=0.1):
        super().__init__()
        self.mha = MultiheadAttention(
            embedding_dim=embedding_dim,
            num_heads=num_heads,
            context_length=context_length,
            is_crossattention=False,
            is_causal=False,
            dropout=dropout
        )
        self.layer_norm1 = nn.LayerNorm(embedding_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim),
            nn.ReLU(),  # original transformer uses ReLU
            nn.Linear(4 * embedding_dim, embedding_dim),
            nn.Dropout(dropout)
        )
        self.layer_norm2 = nn.LayerNorm(embedding_dim)

    def forward(self, x, attention_mask=None):
        # Multi-head attention
        attn_output = self.mha(x, attention_mask=attention_mask)
        x = self.layer_norm1(x + attn_output)  # residual connection + layer norm
        # Feed-forward network
        ffn_output = self.ffn(x)
        x = self.layer_norm2(x + ffn_output)  # residual connection + layer norm
        return x
    

# testig:
encoder_block = EncoderBlock(
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    context_length=context_length,
    dropout=0.1
)


output_encoder_block = encoder_block(batch_x, attention_mask=attention_mask)
print(f"Encoder block output shape: {output_encoder_block.shape}")

Encoder block output shape: torch.Size([2, 6, 768])


## References

 * OpenAI GPT-2 model (TensorFlow): https://github.com/openai/gpt-2/blob/master/src/model.py
 * HuggingFace GPT-2 PyTorch Implementation: https://github.com/huggingface/transformers/blob/main/src/transformers/models/gpt2/modeling_gpt2.py
 * HuggingFace GPT-2 config: https://github.com/huggingface/transformers/blob/main/src/transformers/models/gpt2/configuration_gpt2.py